# 🎛️ BookVoice-AI — Dataset-Tool (Colab GPU)
**v1.0 · Juni 2026**

Transkribiert + schneidet deine Audios auf der **T4-GPU** in ein Trainings-Dataset —
in *exakt* dem Format deines Trainingsraums (`wavs/` + `metadata_train.csv` …).
Schneide-Logik 1:1 aus `_run_training_process`, CSV-Logik aus `training_export`.

**Warum:** large-v3 auf deinem CPU-Server = Stunden. Hier auf T4 = ~10–15 Min für 56 Min Audio.

➡️ Runtime → **T4 GPU**. Dann Zellen 1→4 der Reihe nach.


In [ ]:
#@title 1. Installieren { display-mode: "form" }
print("⏳ Installiere faster-whisper ...")
!pip install -q faster-whisper
import torch
print("✅ ok ·", "CUDA:", torch.cuda.is_available(), "·",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("⚠️  Keine GPU! Runtime → T4 GPU wählen.")


In [ ]:
#@title 2. Einstellungen & Audios laden { display-mode: "form" }
import os, glob, shutil
SPEAKER_NAME = "sufi"      #@param {type:"string"}
LANGUAGE     = "tr"        #@param {type:"string"}
MODEL_SIZE   = "large-v3"  #@param ["tiny","base","small","medium","large-v3"]
EVAL_PROZENT = 0.15        #@param {type:"slider", min:0.05, max:0.4, step:0.05}
MIN_DUR      = 0.34        #@param {type:"number"}
DRIVE_ORDNER = ""          #@param {type:"string"}

IN_DIR  = "/content/audios_in"
OUT_DIR = "/content/dataset"
os.makedirs(IN_DIR, exist_ok=True)

# Audios besorgen: aus Drive-Ordner ODER per Upload
if DRIVE_ORDNER.strip():
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    src = DRIVE_ORDNER if DRIVE_ORDNER.startswith("/content/drive") else f"/content/drive/MyDrive/{DRIVE_ORDNER}"
    exts = (".wav",".mp3",".m4a",".flac",".ogg",".opus",".mp4",".mkv",".webm",".aac")
    n = 0
    for f in glob.glob(os.path.join(src, "*")):
        if f.lower().endswith(exts):
            shutil.copy2(f, IN_DIR); n += 1
    print(f"📁 {n} Dateien aus Drive: {src}")
else:
    from google.colab import files
    print("⬆️  Audios auswählen (Mehrfachauswahl möglich):")
    up = files.upload()
    for name in up:
        shutil.move(name, os.path.join(IN_DIR, name))
    print(f"📁 {len(up)} Dateien hochgeladen")

audios = sorted(glob.glob(os.path.join(IN_DIR, "*")))
assert audios, "❌ Keine Audios gefunden."
print(f"✅ {len(audios)} Audio(s) bereit")


In [ ]:
#@title 3. Transkribieren + Schneiden (GPU) { display-mode: "form" }
import re, subprocess, uuid
from faster_whisper import WhisperModel

device  = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"⏳ Lade Whisper {MODEL_SIZE} ({device}/{compute}) — large-v3 lädt beim 1. Mal ~3GB ...")
model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute)

review = "/content/review"
if os.path.exists(review): shutil.rmtree(review)
os.makedirs(review, exist_ok=True)

def audio_dauer(p):
    r = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
                        "-of","default=noprint_wrappers=1:nokey=1", p], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0

gesamt_clips = 0
for ai, apath in enumerate(audios, 1):
    stem = re.sub(r"[^\w\-]", "_", os.path.splitext(os.path.basename(apath))[0]).strip("_") or "audio"
    print(f"\n🎧 [{ai}/{len(audios)}] {os.path.basename(apath)}")
    # normalisieren 22050 mono
    norm = f"/content/_n_{uuid.uuid4().hex}.wav"
    r = subprocess.run(["ffmpeg","-i",apath,"-ar","22050","-ac","1","-y",norm], capture_output=True)
    if r.returncode != 0 or not os.path.exists(norm):
        print("  ⚠️ FFmpeg-Fehler, übersprungen"); continue
    total_dur = audio_dauer(norm)
    # transkribieren mit Wort-Zeitstempeln
    lang = None if (not LANGUAGE or LANGUAGE == "auto") else LANGUAGE
    segments, info = model.transcribe(norm, language=lang, vad_filter=True, word_timestamps=True)
    words = []
    for seg in segments:
        if seg.words: words.extend(list(seg.words))
    # schneiden (1:1 Server-Logik)
    i = len(glob.glob(os.path.join(review, f"{stem}_*.wav")))
    buffer = 0.2
    satz = ""; satz_start = None; first = True; created = 0
    for idx, w in enumerate(words):
        if first:
            satz_start = w.start
            if idx == 0: satz_start = max(satz_start - buffer, 0)
            else:
                prev_end = words[idx-1].end
                satz_start = max(satz_start - buffer, (prev_end + satz_start)/2)
            satz = w.word; first = False
        else:
            satz += w.word
        if w.word and w.word[-1] in ["!","。",".","?"]:
            next_start = words[idx+1].start if (idx+1 < len(words)) else total_dur
            w_end = min((w.end + next_start)/2, w.end + buffer)
            dur = w_end - satz_start
            if dur >= MIN_DUR and w_end > satz_start:
                cname = f"{stem}_{str(i).zfill(8)}"
                cwav = os.path.join(review, cname + ".wav")
                cut = subprocess.run(["ffmpeg","-i",norm,"-ss",f"{satz_start:.3f}",
                    "-t",f"{dur:.3f}","-ar","22050","-ac","1","-y",cwav], capture_output=True)
                if cut.returncode == 0 and os.path.exists(cwav):
                    txt = satz[1:] if satz.startswith(" ") else satz
                    open(os.path.join(review, cname + ".txt"), "w", encoding="utf-8").write(txt.strip())
                    i += 1; created += 1
            first = True
    os.remove(norm)
    print(f"  ✓ {created} Clips")
    gesamt_clips += created

print(f"\n✅ Gesamt: {gesamt_clips} Clips in {review}")


In [ ]:
#@title 4. Dataset bauen (CSV + ZIP) { display-mode: "form" }
import random, csv as _csv, zipfile

wavs_out = os.path.join(OUT_DIR, "wavs")
if os.path.exists(OUT_DIR): shutil.rmtree(OUT_DIR)
os.makedirs(wavs_out, exist_ok=True)

rows = []
for wav in sorted(glob.glob(os.path.join(review, "*.wav"))):
    stem = os.path.splitext(os.path.basename(wav))[0]
    txtf = os.path.join(review, stem + ".txt")
    text = open(txtf, encoding="utf-8").read().strip() if os.path.exists(txtf) else ""
    if not text: continue
    shutil.copy2(wav, os.path.join(wavs_out, os.path.basename(wav)))
    rows.append([f"wavs/{os.path.basename(wav)}", text, SPEAKER_NAME])

assert rows, "❌ Keine gültigen Clips."
random.shuffle(rows)
pct = max(0.0, min(0.9, EVAL_PROZENT))
n_eval = int(len(rows) * pct)
eval_rows  = sorted(rows[:n_eval], key=lambda x: x[0])
train_rows = sorted(rows[n_eval:], key=lambda x: x[0])

for path_, data in [(os.path.join(OUT_DIR,"metadata_train.csv"), train_rows),
                    (os.path.join(OUT_DIR,"metadata_eval.csv"),  eval_rows)]:
    with open(path_, "w", encoding="utf-8", newline="") as f:
        w = _csv.writer(f, delimiter="|")
        w.writerow(["audio_file","text","speaker_name"])
        w.writerows(data)
open(os.path.join(OUT_DIR,"lang.txt"), "w", encoding="utf-8").write(LANGUAGE + "\n")

# ZIP (flach, wie Server-Export)
zip_path = "/content/dataset.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT_DIR):
        for fn in fs:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, OUT_DIR))

print(f"✅ Dataset: {len(train_rows)} Train · {len(eval_rows)} Eval · Sprache {LANGUAGE}")
print(f"📦 {zip_path}")
print(f"📂 Ordner: {OUT_DIR}  (kannst du direkt im Training-Notebook als DATASET_PATH nutzen)")
from google.colab import files as _f
_f.download(zip_path)
